1. Model: Qwen3-VL-8B vision with Unsloth (4-bit)

In [ ]:
!pip install -U "unsloth>=2024.10.0" "transformers>=4.57.0" datasets pillow accelerate bitsandbytes trl

2. Dataset: use your JSONL + real images

You already have JSONL rows like (simplified):

In [10]:
from datasets import load_dataset
from PIL import Image
import os

DATA_PATH   = "training_dataset.jsonl"   # your JSONL
IMAGES_ROOT = "."                           # root so that url is relative, change if needed

dataset = load_dataset(
    "json",
    data_files={"train": DATA_PATH},
)["train"]


Convert messages into Unsloth vision format

In [11]:
# Convert HF Dataset → Python list (required by SFTTrainer for vision)
train_data = dataset.to_list()

print("Loaded examples:", len(train_data))


Loaded examples: 221


In [16]:
print(train_data[0]["messages"][2])
ex = train_data[0]
for msg in ex["messages"]:
    print("ROLE:", msg["role"])
    for block in msg["content"]:
        print("   ", block["type"], type(block.get("image") or block.get("text")))


{'role': 'assistant', 'content': [{'type': 'text', 'text': '{"PatientSurname": {"value": "T108", "bbox": [0.043954, 0.218127, 0.233253, 0.240445]}, "PatientForeName": {"value": "T108", "bbox": [0.239541, 0.220393, 0.436699, 0.241599]}, "DateOfBirth": {"value": "1/5/61", "bbox": [0.04081, 0.242711, 0.28688, 0.265028]}, "Gender": {"value": "fEMale", "bbox": [0.296312, 0.240445, 0.507678, 0.263916]}, "Ethnicity": {"value": "Caucasian", "bbox": [0.04081, 0.26165, 0.27896, 0.285079]}, "Address": {"value": "108 Green Street", "bbox": [0.037666, 0.285079, 0.308948, 0.308551]}, "HospitalNumber": {"value": "0000108", "bbox": [0.039238, 0.311885, 0.321584, 0.32749]}, "NHSNumber": {"value": null, "bbox": [0.037666, 0.330868, 0.348368, 0.350919]}, "LandlineNumber": {"value": null, "bbox": [0.04081, 0.353185, 0.338936, 0.373236]}, "MobileNumber": {"value": "07936411198", "bbox": [0.04081, 0.374391, 0.362576, 0.396708]}, "ConsentToText": {"value": "Yes", "bbox": [0.042382, 0.394442, 0.495103, 0.4312

3. Supervised fine-tuning (SFT) with vision collator

Unsloth provides a vision data collator + standard SFTTrainer for this.

In [17]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))


CUDA available: False


### Training model

In [ ]:
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
import torch



# ---------------------------------------------------------
# 1) Load base model (Qwen3-VL-8B) in 4-bit
# ---------------------------------------------------------
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit",
    load_in_4bit = True,
    dtype = None,                      # auto dtype
    use_gradient_checkpointing = "unsloth"
)

print("Model loaded.")


# ---------------------------------------------------------
# 2) Add LoRA adapters
# ---------------------------------------------------------
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r           = 16,
    lora_alpha  = 16,
    lora_dropout= 0.0,
    bias        = "none",
    target_modules = "all-linear",
    use_rslora  = False,              # Qwen3-VL does NOT benefit from RSLoRA
    loftq_config= None,
)


FastVisionModel.for_training(model)
print("LoRA enabled.")

param_device = next(model.parameters()).device
print("Model is on:", param_device)




# ---------------------------------------------------------
# 3) Vision-aware data collator
# ---------------------------------------------------------
data_collator = UnslothVisionDataCollator(model, tokenizer)


# ---------------------------------------------------------
# 4) Training config (SFT)
# ---------------------------------------------------------
train_args = SFTConfig(
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate              = 2e-4,
    max_steps                  = 180,      # <= ~3 epochs
    warmup_ratio               = 0.03,
    logging_steps              = 10,
    save_steps                 = 200,
    weight_decay               = 0.01,
    bf16                       = is_bf16_supported(),
    output_dir                 = "qwen3-vl-8b-referral-forms",
    remove_unused_columns      = False,
    max_seq_length             = 2048,
    dataset_text_field         = "",
    dataset_kwargs             = {"skip_prepare_dataset": True},
)


# ---------------------------------------------------------
# 5) Trainer
# ---------------------------------------------------------
trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    args          = train_args,
    train_dataset = train_data,       # IMPORTANT: must be a Python list
    data_collator = data_collator,
)

print("Training…")
trainer.train()

print("Training complete.")


In [ ]:
!nvidia-smi
print("Model device:", next(model.parameters()).device)

This will fine-tune only the LoRA adapters while keeping Qwen3-VL’s vision + language backbone frozen, in 4-bit, which is what lets this run on modest GPUs.

4. Inference on a new referral form

Once training is done:

In [ ]:
from PIL import Image
import torch

FastVisionModel.for_inference(model)

def run_inference(image_path, instruction):
    image = Image.open(image_path).convert("RGB")

    # Build a single-turn vision chat
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": instruction},
            ],
        }
    ]

    # Qwen3-VL template is already inside tokenizer from Unsloth 
    input_text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = False,
    )

    inputs = tokenizer(
        image,
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 512,
            temperature     = 0.2,
            top_p           = 0.8,
        )

    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text

test_json = run_inference(
    "dataset/images_by_page/page_1_1/gp-referral-cancer-colorectal IOV108_page_1.png",
    "Extract the referral form fields and return STRICT valid JSON only."
)
print(test_json)


You can then json.loads(test_json) and validate / post-process.

5. Saving the fine-tuned Qwen3-VL model
Save LoRA adapters

In [ ]:
adapter_dir = "qwen3-vl-8b-referral-forms-lora"

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print("Saved LoRA adapters to", adapter_dir)

(Optional) Merge LoRA into full weights for deployment

In [ ]:
merged_dir = "qwen3-vl-8b-referral-forms-merged"

# merged = FastVisionModel.merge_and_unload(model)
model.save_pretrained_merged(merged_dir, tokenizer)
tokenizer.save_pretrained(merged_dir)

print("Saved merged model to", merged_dir)
model.push_to_hub_merged("sophy/finetuned-qwen-referrals", tokenizer, token="")


In [ ]:
# Save to 16bit GGUF

# model.save_pretrained_gguf("unsloth_qwen3-vl-8b-referral-forms", tokenizer, quantization_method = "f16")
# model.push_to_hub_gguf("hf/unsloth_finetune_qwen3-vl-8b-referral-forms", tokenizer, quantization_method = "f16", token = "")

The merged model can be served with vLLM, llama.cpp (via GGUF exports), Docker, etc. – the same infra Unsloth shows for Qwen3-VL.

### Testing trained HF Model

In [ ]:
from unsloth import FastVisionModel

MODEL_REPO = "sophy/finetuned-qwen-referrals"

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_REPO,
    load_in_4bit = True,
)

FastVisionModel.for_inference(model)


==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.402 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████████████| 4/4 [00:26<00:00,  6.73s/it]


✅ Base + LoRA ready to merge


In [11]:
from PIL import Image
import torch

image_path = "test.png"
image = Image.open(image_path).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text",  "text": "Extract all fields such as patient details, gp details including InterpreterRequired, DateOfReferral, DateOOfDecisionToRefer, and ReasonFoRrReferral and return JSON."},
        ],
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)

inputs = tokenizer(image, prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=2000,
        temperature=0.1,
        top_p=0.9,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))


user
Extract all fields such as patient details, gp details including InterpreterRequired, DateOfReferral, DateOOfDecisionToRefer, and ReasonFoRrReferral and return JSON.
assistant
{"PatientName": {"value": "Ms Green", "bbox": {"x": 136.2, "y": 231.8, "width": 446.1, "height": 39.2}}, "NHSNumber": {"value": "01", "bbox": {"x": 133.6, "y": 271, "width": 448.7, "height": 36.5}}, "DateOfBirth": {"value": "6/1/71", "bbox": {"x": 133.6, "y": 286.5, "width": 297.4, "height": 44.4}}, "Gender": {"value": "female", "bbox": {"x": 133.6, "y": 328.3, "width": 464.4, "height": 34}}, "Address": {"value": "5 Timbuktu Street", "bbox": {"x": 133.6, "y": 367.6, "width": 467, "height": 39.1}}, "Telephone": {"value": "", "bbox": {"x": 133.6, "y": 409.3, "width": 456.6, "height": 34}}, "MobileNumber": {"value": "", "bbox": {"x": 133.6, "y": 443.5, "width": 464.4, "height": 34}}, "Email": {"value": "", "bbox": {"x": 133.6, "y": 500.9, "width": 454, "height": 23.5}}, "ConsentToText": {"value": "Yes", "bbox":

### GGUF

In [18]:
# BASE_MODEL   = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
MODEL_REPO = "sophy/finetuned-qwen-referrals"

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_REPO,
    load_in_4bit = True,
    trust_remote_code = True
)



Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.6: Fast Qwen3_Vl patching. Transformers: 4.57.2.
   \\   /|    NVIDIA RTX A6000. Num GPUs = 1. Max memory: 47.402 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Qwen3_Vl does not support SDPA - switching to fast eager.


Loading checkpoint shards: 100%|██████████████████| 4/4 [00:19<00:00,  4.77s/it]


In [22]:
import os
os.environ["HF_HUB_OFFLINE"]= "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"

In [ ]:
model.save_pretrained_gguf(
    "qwen3vl_referrals_gguf",
    tokenizer= tokenizer,
    quantization_method="q4_k_m"
)